In [31]:
!pip -q install duckdb

In [32]:
import duckdb

con = duckdb.connect()

print("DuckDB Ready!")

DuckDB Ready!


In [33]:
from huggingface_hub import hf_hub_download

In [34]:
sample = hf_hub_download(
    repo_id="FlyRank/internship-warehouse",
    filename="fact_content_daily_performance_sample.parquet",
    repo_type="dataset"
)

print(sample)

/root/.cache/huggingface/hub/datasets--FlyRank--internship-warehouse/snapshots/50cbf7c3909d07be4d1b5906b4d09e882e5acbf2/fact_content_daily_performance_sample.parquet


In [35]:
import duckdb

con = duckdb.connect()

con.sql(f"""
SELECT *
FROM read_parquet('{sample}')
LIMIT 5
""").df()

,report_date,client_hash_id,content_hash_id,client_has_gsc,client_has_ga4,gsc_data_available,ga4_data_available,gsc_impressions,gsc_clicks,gsc_sum_position,...,sessions_ai,ai_chatgpt,ai_perplexity,ai_gemini,ai_copilot,ai_claude,ai_meta,ai_other,scroll_events,month
0,2026-06-01,client_3ffa76342f366962,content_1a6296faee432dae,True,True,False,False,0,0,0,...,0,0,0,0,0,0,0,0,0,2026-06
1,2026-06-01,client_3ffa76342f366962,content_73f21e612565035a,True,True,False,False,0,0,0,...,0,0,0,0,0,0,0,0,0,2026-06
2,2026-06-01,client_3ffa76342f366962,content_5a5be514ff559598,True,True,False,False,0,0,0,...,0,0,0,0,0,0,0,0,0,2026-06
3,2026-06-01,client_3ffa76342f366962,content_05b377d0c8a5cfd8,True,True,False,False,0,0,0,...,0,0,0,0,0,0,0,0,0,2026-06
4,2026-06-01,client_3ffa76342f366962,content_dc34c661d63e55a9,True,True,False,False,0,0,0,...,0,0,0,0,0,0,0,0,0,2026-06


In [36]:
con.sql(f"""
DESCRIBE
SELECT *
FROM read_parquet('{sample}')
""").df()

,column_name,column_type,null,key,default,extra
0,report_date,DATE,YES,None,None,None
1,client_hash_id,VARCHAR,YES,None,None,None
2,content_hash_id,VARCHAR,YES,None,None,None
3,client_has_gsc,BOOLEAN,YES,None,None,None
4,client_has_ga4,BOOLEAN,YES,None,None,None
5,gsc_data_available,BOOLEAN,YES,None,None,None
6,ga4_data_available,BOOLEAN,YES,None,None,None
7,gsc_impressions,BIGINT,YES,None,None,None
8,gsc_clicks,BIGINT,YES,None,None,None
9,gsc_sum_position,BIGINT,YES,None,None,None


In [37]:
con.sql(f"""
SELECT COUNT(*) AS total_rows
FROM read_parquet('{sample}')
""").df()

,total_rows
0,11694072


## Unit of analysis + time window

One row represents one content page for one client on one report date.

I am using the `fact_content_daily_performance` table.

The sample data covers **2026-06-01 to 2026-06-30**.

My goal is to identify pages that should be prioritized for optimization based on their search performance.

I deliberately exclude future information and any label-derived columns to avoid data leakage.

In [38]:
con.sql(f"""
SELECT
    MIN(report_date) AS start_date,
    MAX(report_date) AS end_date
FROM read_parquet('{sample}')
""").df()

,start_date,end_date
0,2026-06-01,2026-06-30


## Features

- gsc_impressions
- gsc_clicks
- gsc_avg_position
- ga4_sessions
- ga4_engaged_sessions

## Label / Proxy

Pages that have impressions but receive very few or zero clicks are considered higher optimization priority.

## Context

- report_date
- month
- client_hash_id
- content_hash_id

## Excluded

Future information and any label-derived features are excluded because they would cause data leakage.

In [39]:
con.sql(f"""
SELECT
    gsc_impressions,
    gsc_clicks,
    gsc_avg_position,
    ga4_sessions,
    ga4_engaged_sessions
FROM read_parquet('{sample}')
LIMIT 5
""").df()

,gsc_impressions,gsc_clicks,gsc_avg_position,ga4_sessions,ga4_engaged_sessions
0,0,0,NaN,0,0
1,0,0,NaN,0,0
2,0,0,NaN,0,0
3,0,0,NaN,0,0
4,0,0,NaN,0,0


## Verification

The following queries verify the data contract.

- One row represents one page for one client on one report date.
- The sample contains 11,694,072 rows.
- The sample covers 2026-06-01 to 2026-06-30.
- 3,878,937 rows have Search Console data available.

In [40]:
print("Query 1 - Total Rows")
display(con.sql(f"""
SELECT COUNT(*) AS total_rows
FROM read_parquet('{sample}')
""").df())

print("Query 2 - Date Range")
display(con.sql(f"""
SELECT
MIN(report_date) AS start_date,
MAX(report_date) AS end_date
FROM read_parquet('{sample}')
""").df())

print("Query 3 - GSC Available")
display(con.sql(f"""
SELECT COUNT(*) AS available_rows
FROM read_parquet('{sample}')
WHERE gsc_data_available IS TRUE
""").df())

Query 1 - Total Rows


,total_rows
0,11694072


Query 2 - Date Range


,start_date,end_date
0,2026-06-01,2026-06-30


Query 3 - GSC Available


,available_rows
0,3878937


## Five-feature frame

| Feature | Available at decision time because... |
|----------|---------------------------------------|
| gsc_impressions | Search Console already reports impressions before optimization decisions. |
| gsc_clicks | Clicks are already observed for the reporting period. |
| gsc_avg_position | Search position is already known from Search Console. |
| ga4_sessions | Analytics sessions are already collected. |
| ga4_engaged_sessions | User engagement has already been measured. |

## Leakage demonstration

A label-derived feature can make model performance appear unrealistically high because it contains information that would not be available at prediction time.

Such features must be removed before training to obtain an honest evaluation.

In [41]:
print("Leakage demonstration completed conceptually. Label-derived features must be excluded from training.")

Leakage demonstration completed conceptually. Label-derived features must be excluded from training.


## Data limits

This notebook uses the June 2026 sample dataset for testing queries and building features.

The sample is useful for understanding the data structure, but it should not be used for developing future-looking labels.

Some rows do not contain Search Console or Google Analytics data.

External factors such as seasonality, competitor actions and search algorithm updates are not included.

The results should therefore be treated as decision-support rather than proof of causation.

## Self-check

-  Unit of analysis defined
-  Features identified
-  Three verification queries completed
-  Five-feature frame included
-  Leakage explained
-  Data limitations described
-  Notebook ready for GitHub submission